In [ ]:
!nvidia-smi

Tue Oct 13 09:44:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap --format=csv

name, compute_cap
Tesla T4, 7.5


In [ ]:
%%writefile ex1_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    // each block runs only one addition
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

// Track CPU (wall-clock) time, in seconds
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;             // host copies of a, b, c
    int *d_a, *d_b, *d_c;       // device copies of a, b, c
    int size = N * sizeof(int);
    double dStart, dElaps;

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate space for host copies of a, b, c and set up input values
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // initialize timer
    dStart = cpuSecond();

    // Launch add() kernel on GPU with N blocks
    add<<<N, 1>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize();    // wait for the GPU to finish, THEN stop the clock

    dElaps = cpuSecond() - dStart;

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < 5; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    // Check the GPU's answers against the CPU
    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    printf("Time taken - %.7f\n", dElaps);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Overwriting ex1_timed.cu


In [ ]:
!nvcc -arch=sm_75 ex1_timed.cu -o ex1_timed && ./ex1_timed

83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
Verification: 0 mismatches out of 512
Time taken - 0.0001030


Question 1: What happens to the time if cudaDeviceSynchronize() is removed?

The measured time becomes much smaller because the CPU timer stops after the asynchronous kernel launch instead of waiting for the GPU to finish its computation. The reported time is therefore not the actual kernel execution time.

Question 2: Does the answer still verify correctly?

It may still verify correctly because the subsequent device-to-host cudaMemcpy() synchronizes with the necessary GPU work before the results are read. However, the timer no longer measures the complete kernel execution.

Question 3: Why must cudaDeviceSynchronize() be placed before stopping the clock?

CUDA kernel launches are asynchronous. cudaDeviceSynchronize() makes the CPU wait until previously issued GPU work has completed. Therefore, placing it before the ending time measurement ensures that the measured interval includes the kernel execution.


In [ ]:
%%writefile ex2_check.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>
#include <cuda_runtime.h>

#define N 512

#define CHECK(call)                                                   \
    do {                                                              \
        const cudaError_t error = (call);                             \
        if (error != cudaSuccess) {                                   \
            printf("Error: %s:%d, ", __FILE__, __LINE__);             \
            printf("code:%d, reason: %s\n", error,                    \
                   cudaGetErrorString(error));                        \
            exit(1);                                                  \
        }                                                             \
    } while (0)

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++) {
        x[i] = rand() % 100;
    }
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);
    double dStart, dElaps;

    CHECK(cudaMalloc((void **)&d_a, size));
    CHECK(cudaMalloc((void **)&d_b, size));
    CHECK(cudaMalloc((void **)&d_c, size));

    a = (int *)malloc(size);
    b = (int *)malloc(size);
    c = (int *)malloc(size);

    if (a == NULL || b == NULL || c == NULL) {
        printf("Host memory allocation failed\n");
        free(a);
        free(b);
        free(c);
        cudaFree(d_a);
        cudaFree(d_b);
        cudaFree(d_c);
        return 1;
    }

    random_ints(a, N);
    random_ints(b, N);

    CHECK(cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice));

    dStart = cpuSecond();

    // Experiment A: deliberately use too many threads per block
    add<<<N, 2048>>>(d_a, d_b, d_c);

    cudaError_t launchError = cudaGetLastError();

    if (launchError != cudaSuccess) {
        printf("Launch error: code:%d, reason: %s\n",
               launchError, cudaGetErrorString(launchError));

        free(a);
        free(b);
        free(c);
        cudaFree(d_a);
        cudaFree(d_b);
        cudaFree(d_c);
        return 0;
    }

    CHECK(cudaDeviceSynchronize());

    dElaps = cpuSecond() - dStart;

    CHECK(cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost));

    for (int r = 0; r < 5; r++) {
        printf("%d + %d = %d\n", a[r], b[r], c[r]);
    }

    int errors = 0;

    for (int r = 0; r < N; r++) {
        if (c[r] != a[r] + b[r]) {
            errors++;
        }
    }

    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a);
    free(b);
    free(c);

    CHECK(cudaFree(d_a));
    CHECK(cudaFree(d_b));
    CHECK(cudaFree(d_c));

    return 0;
}


Overwriting ex2_check.cu


In [ ]:
!nvcc -arch=sm_75 ex2_check.cu -o ex2_check && ./ex2_check

83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
Verification: 0 mismatches out of 512
Time taken - 0.0001109


In [25]:
%%writefile ex1_no_sync.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    // each block runs only one addition
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

// Track CPU (wall-clock) time, in seconds
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;             // host copies of a, b, c
    int *d_a, *d_b, *d_c;       // device copies of a, b, c
    int size = N * sizeof(int);
    double dStart, dElaps;

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate space for host copies of a, b, c and set up input values
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // initialize timer
    dStart = cpuSecond();

    // Launch add() kernel on GPU with N blocks
    add<<<N, 1>>>(d_a, d_b, d_c);

    dElaps = cpuSecond() - dStart;

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < 5; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    // Check the GPU's answers against the CPU
    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    printf("Time taken - %.7f\n", dElaps);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing ex1_no_sync.cu


In [26]:
!nvcc -arch=sm_75 ex1_no_sync.cu -o ex1_no_sync && ./ex1_no_sync

83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
Verification: 0 mismatches out of 512
Time taken - 0.0001209


Experiment A — Too many threads per block

This experiment tests kernel launch error detection. The launch configuration is invalid because a block cannot contain 2,048 threads on the T4 GPU. CHECK(cudaGetLastError()) detects the launch error and stops the program.

In [ ]:
!nvcc -arch=sm_75 ex2_check.cu -o ex2_check && ./ex2_check

Error: ex2_check.cu:40, code:2, reason: out of memory


Experiment B — Impossible allocation

This experiment tests CUDA memory-allocation error handling. The program requests 2^50 bytes, which is far beyond the available GPU memory, so the allocation fails.

In [ ]:
!nvcc -arch=sm_75 ex1_timed.cu -o ex1_timed && ./ex1_timed

83 + 77 = 0
86 + 34 = 0
77 + 90 = 0
15 + 26 = 0
93 + 24 = 0
Verification: 512 mismatches out of 512
Time taken - 0.0001252


Experiment C — Try the invalid launch without CHECK()

Without CHECK(cudaGetLastError()), the program does not immediately report the launch error clearly. The kernel fails to execute, and the verification step may detect incorrect results. This experiment demonstrates why CUDA error checking is important for identifying and debugging kernel launch failures.

In [ ]:

%%writefile ex3_picture.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>
#include <cuda_runtime.h>

#define WIDTH 800
#define HEIGHT 600

#define CHECK(call)                                                   \
    do {                                                              \
        const cudaError_t error = (call);                             \
        if (error != cudaSuccess) {                                   \
            printf("Error: %s:%d, ", __FILE__, __LINE__);             \
            printf("code:%d, reason: %s\n", error,                    \
                   cudaGetErrorString(error));                        \
            exit(1);                                                  \
        }                                                             \
    } while (0)

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

__global__ void PictureKernel(unsigned char *d_Pin,
                              unsigned char *d_Pout,
                              int height, int width) {
    int Row = blockIdx.y * blockDim.y + threadIdx.y;
    int Col = blockIdx.x * blockDim.x + threadIdx.x;

    if (Row < height && Col < width) {
        int v = 2 * d_Pin[Row * width + Col];
        d_Pout[Row * width + Col] = (v > 255) ? 255 : v;
    }
}

int main(void) {
    size_t count = (size_t)WIDTH * HEIGHT;
    size_t size = count * sizeof(unsigned char);

    unsigned char *h_in, *h_out;
    unsigned char *d_in, *d_out;
    double dStart, dElaps;

    h_in = (unsigned char *)malloc(size);
    h_out = (unsigned char *)malloc(size);

    if (h_in == NULL || h_out == NULL) {
        printf("Host memory allocation failed\n");
        free(h_in);
        free(h_out);
        return 1;
    }

    CHECK(cudaMalloc((void **)&d_in, size));
    CHECK(cudaMalloc((void **)&d_out, size));

    // TODO 1: Initialize pixels.
    for (size_t i = 0; i < count; i++) {
        h_in[i] = rand() % 256;
    }

    CHECK(cudaMemcpy(d_in, h_in, size, cudaMemcpyHostToDevice));

    // TODO 2: Threads per block.
    dim3 threadsPerBlock(16, 16);

    // TODO 3: Grid dimensions, rounded up.
    dim3 numBlocks(
        (WIDTH + threadsPerBlock.x - 1) / threadsPerBlock.x,
        (HEIGHT + threadsPerBlock.y - 1) / threadsPerBlock.y
    );

    printf("Grid: %d x %d blocks of %d x %d threads\n",
           numBlocks.x, numBlocks.y,
           threadsPerBlock.x, threadsPerBlock.y);

    dStart = cpuSecond();

    // TODO 4: Launch kernel.
    PictureKernel<<<numBlocks, threadsPerBlock>>>(
        d_in, d_out, HEIGHT, WIDTH
    );

    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());

    dElaps = cpuSecond() - dStart;

    CHECK(cudaMemcpy(h_out, d_out, size, cudaMemcpyDeviceToHost));

    // TODO 5: Print original 50 x 50 portion.
    printf("\nOriginal picture (top-left 50 x 50):\n");

    for (int row = 0; row < 50; row++) {
        for (int col = 0; col < 50; col++) {
            printf("%3d ", (int)h_in[row * WIDTH + col]);
        }
        printf("\n");
    }

    printf("\nUpdated picture (top-left 50 x 50):\n");

    for (int row = 0; row < 50; row++) {
        for (int col = 0; col < 50; col++) {
            printf("%3d ", (int)h_out[row * WIDTH + col]);
        }
        printf("\n");
    }

    // TODO 6: Verify all pixels and count capped values.
    int errors = 0;
    size_t capped = 0;

    for (size_t i = 0; i < count; i++) {
        int expected = 2 * (int)h_in[i];

        if (expected > 255) {
            expected = 255;
            capped++;
        }

        if ((int)h_out[i] != expected) {
            errors++;
        }
    }

    printf("\nVerification: %d mismatches out of %zu\n",
           errors, count);
    printf("Pixels capped at 255: %zu\n", capped);
    printf("Time taken - %.7f\n", dElaps);

    free(h_in);
    free(h_out);

    CHECK(cudaFree(d_in));
    CHECK(cudaFree(d_out));

    return 0;
}

Writing ex3_picture.cu


In [ ]:
!nvcc -arch=sm_75 ex3_picture.cu -o ex3_picture && ./ex3_picture

Grid: 50 x 38 blocks of 16 x 16 threads

Original picture (top-left 50 x 50):
103 198 105 115  81 255  74 236  41 205 186 171 242 251 227  70 124 194  84 248  27 232 231 141 118  90  46  99  51 159 201 154 102  50  13 183  49  88 163  90  37  93   5  23  88 233  94 212 171 178 
 96  34  10 167  77  49 181  91   3 160  13  34  13  71  93 205 155 135 120  86 213 112  76 156 134 234  15 152 242 235 156  83  13 167 250  90 216 176 181 219  80 194 253  93   9  90  42 165 226 163 
210 229 220  71  16 152 132 214 161  59  36  81  31  29 107 245  90 125  16 216  23 252 165  61 140  36 239 252 218 206  78 172 179  42 243 196 195 119 154 100 178 190 181 209 219  32 198  53 157 214 
158 212 187 189 162 172 238 192 141 116  54 141  24 225  34 225 154   4  34 178 109 178 216 130 145 231 176 222 132 115 155  34  71  86 223 233   2 205 169 143  65 224  28  90 193  63  59  91  67  93 
219 240 232   2 175 227 102  92  27 167 209 145 153  42 245 250 103 153 124 186 196 109  26  59 117 143  79  87 135 18

In [ ]:

%%writefile ex4_volume.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>
#include <cuda_runtime.h>

#define NX 500
#define NY 500
#define NZ 500

#define CHECK(call)                                                   \
    do {                                                              \
        const cudaError_t error = (call);                             \
        if (error != cudaSuccess) {                                   \
            printf("Error: %s:%d, ", __FILE__, __LINE__);             \
            printf("code:%d, reason: %s\n", error,                    \
                   cudaGetErrorString(error));                        \
            exit(1);                                                  \
        }                                                             \
    } while (0)

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

// Element-wise multiplication:
// C[z][y][x] = A[z][y][x] * B[z][y][x]
__global__ void mulElem3D(int *a, int *b, int *c,
                          int nx, int ny, int nz) {

    // TODO 1: Calculate thread coordinates.
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    int z = blockIdx.z * blockDim.z + threadIdx.z;

    // TODO 2 and 3: Bounds check and flat index.
    if (x < nx && y < ny && z < nz) {
        size_t idx = ((size_t)z * ny + y) * nx + x;
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, size_t size) {
    for (size_t i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    size_t count = (size_t)NX * NY * NZ;
    size_t size = count * sizeof(int);

    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    double dStart, dElaps;

    // Allocate device arrays.
    CHECK(cudaMalloc((void **)&d_a, size));
    CHECK(cudaMalloc((void **)&d_b, size));
    CHECK(cudaMalloc((void **)&d_c, size));

    // Allocate host arrays.
    a = (int *)malloc(size);
    b = (int *)malloc(size);
    c = (int *)malloc(size);

    if (a == NULL || b == NULL || c == NULL) {
        printf("Host memory allocation failed\n");

        free(a);
        free(b);
        free(c);

        cudaFree(d_a);
        cudaFree(d_b);
        cudaFree(d_c);

        return 1;
    }

    random_ints(a, count);
    random_ints(b, count);

    CHECK(cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice));

    // 8 x 8 x 8 = 512 threads per block.
    dim3 threadsPerBlock(8, 8, 8);

    // TODO 4: Calculate the 3D grid dimensions.
    dim3 numBlocks(
        (NX + threadsPerBlock.x - 1) / threadsPerBlock.x,
        (NY + threadsPerBlock.y - 1) / threadsPerBlock.y,
        (NZ + threadsPerBlock.z - 1) / threadsPerBlock.z
    );

    printf("Grid: %d x %d x %d blocks\n",
           numBlocks.x, numBlocks.y, numBlocks.z);

    dStart = cpuSecond();

    // TODO 5: Launch the 3D kernel.
    mulElem3D<<<numBlocks, threadsPerBlock>>>(
        d_a, d_b, d_c, NX, NY, NZ
    );

    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());

    dElaps = cpuSecond() - dStart;

    CHECK(cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost));

    // TODO 6: Print the last 1000 elements in memory order.
    printf("\nLast 1000 elements of C:\n");

    for (size_t i = count - 1000; i < count; i++) {
        size_t z = i / ((size_t)NY * NX);
        size_t remainder = i % ((size_t)NY * NX);
        size_t y = remainder / NX;
        size_t x = remainder % NX;

        printf("C[%zu][%zu][%zu] = %d x %d = %d\n",
               z, y, x, a[i], b[i], c[i]);
    }

    // TODO 7: Verify every element.
    size_t errors = 0;

    for (size_t i = 0; i < count; i++) {
        if (c[i] != a[i] * b[i])
            errors++;
    }

    printf("\nVerification: %zu mismatches out of %zu\n",
           errors, count);

    printf("Time taken - %.7f\n", dElaps);

    free(a);
    free(b);
    free(c);

    CHECK(cudaFree(d_a));
    CHECK(cudaFree(d_b));
    CHECK(cudaFree(d_c));

    return 0;
}

Writing ex4_volume.cu


In [ ]:
!nvcc -arch=sm_75 ex4_volume.cu -o ex4_volume && ./ex4_volume

Grid: 63 x 63 x 63 blocks

Last 1000 elements of C:
C[499][498][0] = 76 x 15 = 1140
C[499][498][1] = 38 x 93 = 3534
C[499][498][2] = 34 x 11 = 374
C[499][498][3] = 85 x 47 = 3995
C[499][498][4] = 3 x 38 = 114
C[499][498][5] = 43 x 82 = 3526
C[499][498][6] = 16 x 52 = 832
C[499][498][7] = 28 x 23 = 644
C[499][498][8] = 32 x 28 = 896
C[499][498][9] = 80 x 98 = 7840
C[499][498][10] = 69 x 81 = 5589
C[499][498][11] = 45 x 79 = 3555
C[499][498][12] = 30 x 2 = 60
C[499][498][13] = 41 x 13 = 533
C[499][498][14] = 30 x 81 = 2430
C[499][498][15] = 30 x 74 = 2220
C[499][498][16] = 41 x 13 = 533
C[499][498][17] = 14 x 18 = 252
C[499][498][18] = 83 x 20 = 1660
C[499][498][19] = 13 x 28 = 364
C[499][498][20] = 86 x 17 = 1462
C[499][498][21] = 13 x 6 = 78
C[499][498][22] = 88 x 57 = 5016
C[499][498][23] = 71 x 29 = 2059
C[499][498][24] = 28 x 17 = 476
C[499][498][25] = 78 x 2 = 156
C[499][498][26] = 50 x 37 = 1850
C[499][498][27] = 86 x 40 = 3440
C[499][498][28] = 28 x 35 = 980
C[499][498][29] = 15 

Exercise 1(b) — Time the four Lab 9 programs

Lab 9 Exercise 3 — timed copy

In [1]:
%%writefile ex3_blocks_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>


double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

#define N 512

__global__ void add(int *a, int *b, int *c) {
    int i = blockIdx.x;
    c[i] = a[i] + b[i];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);
    double dStart, dElaps;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size);
    b = (int *)malloc(size);
    c = (int *)malloc(size);

    random_ints(a, N);
    random_ints(b, N);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    dStart = cpuSecond();
    add<<<N, 1>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    cudaDeviceSynchronize();
    dElaps = cpuSecond() - dStart;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int i = 0; i < N; i++)
        printf("%d + %d = %d\n", a[i], b[i], c[i]);

    int errors = 0;
    for (int i = 0; i < N; i++)
        if (c[i] != a[i] + b[i]) errors++;

    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a);
    free(b);
    free(c);
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}

Writing ex3_blocks_timed.cu


In [2]:
!nvcc -arch=sm_75 ex3_blocks_timed.cu -o ex3_blocks_timed

In [19]:
!./ex3_blocks_timed

83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
35 + 57 = 92
86 + 14 = 100
92 + 68 = 160
49 + 5 = 54
21 + 58 = 79
62 + 12 = 74
27 + 86 = 113
90 + 0 = 90
59 + 46 = 105
63 + 26 = 89
26 + 94 = 120
40 + 16 = 56
26 + 52 = 78
72 + 78 = 150
36 + 29 = 65
11 + 46 = 57
68 + 90 = 158
67 + 47 = 114
29 + 70 = 99
82 + 51 = 133
30 + 80 = 110
62 + 31 = 93
23 + 93 = 116
67 + 57 = 124
35 + 27 = 62
29 + 12 = 41
2 + 86 = 88
22 + 14 = 36
58 + 55 = 113
69 + 12 = 81
67 + 90 = 157
93 + 12 = 105
56 + 79 = 135
11 + 10 = 21
42 + 69 = 111
29 + 89 = 118
73 + 74 = 147
21 + 55 = 76
19 + 41 = 60
84 + 20 = 104
37 + 33 = 70
98 + 87 = 185
24 + 88 = 112
15 + 38 = 53
70 + 66 = 136
13 + 70 = 83
26 + 84 = 110
91 + 56 = 147
80 + 17 = 97
56 + 6 = 62
73 + 60 = 133
62 + 49 = 111
70 + 37 = 107
96 + 5 = 101
81 + 59 = 140
5 + 17 = 22
25 + 18 = 43
84 + 45 = 129
27 + 83 = 110
36 + 73 = 109
5 + 58 = 63
46 + 73 = 119
29 + 37 = 66
13 + 89 = 102
57 + 83 = 140
24 + 7 = 31
95 + 78 = 173
82 + 57 = 139
45 + 14 = 59
14 +

Lab 9 Exercise 4 — timed copy

In [4]:
%%writefile ex4_threads_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>


double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

#define N 512

__global__ void addT(int *a, int *b, int *c) {
    int i = threadIdx.x;
    c[i] = a[i] + b[i];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);
    double dStart, dElaps;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size);
    b = (int *)malloc(size);
    c = (int *)malloc(size);

    random_ints(a, N);
    random_ints(b, N);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    dStart = cpuSecond();
    addT<<<1, N>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    cudaDeviceSynchronize();
    dElaps = cpuSecond() - dStart;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int i = 0; i < N; i++)
        printf("%d) %d + %d = %d\n", i, a[i], b[i], c[i]);

    int errors = 0;
    for (int i = 0; i < N; i++)
        if (c[i] != a[i] + b[i]) errors++;

    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a);
    free(b);
    free(c);
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}

Writing ex4_threads_timed.cu


In [5]:
!nvcc -arch=sm_75 ex4_threads_timed.cu -o ex4_threads_timed

In [20]:
!./ex4_threads_timed

0) 83 + 77 = 160
1) 86 + 34 = 120
2) 77 + 90 = 167
3) 15 + 26 = 41
4) 93 + 24 = 117
5) 35 + 57 = 92
6) 86 + 14 = 100
7) 92 + 68 = 160
8) 49 + 5 = 54
9) 21 + 58 = 79
10) 62 + 12 = 74
11) 27 + 86 = 113
12) 90 + 0 = 90
13) 59 + 46 = 105
14) 63 + 26 = 89
15) 26 + 94 = 120
16) 40 + 16 = 56
17) 26 + 52 = 78
18) 72 + 78 = 150
19) 36 + 29 = 65
20) 11 + 46 = 57
21) 68 + 90 = 158
22) 67 + 47 = 114
23) 29 + 70 = 99
24) 82 + 51 = 133
25) 30 + 80 = 110
26) 62 + 31 = 93
27) 23 + 93 = 116
28) 67 + 57 = 124
29) 35 + 27 = 62
30) 29 + 12 = 41
31) 2 + 86 = 88
32) 22 + 14 = 36
33) 58 + 55 = 113
34) 69 + 12 = 81
35) 67 + 90 = 157
36) 93 + 12 = 105
37) 56 + 79 = 135
38) 11 + 10 = 21
39) 42 + 69 = 111
40) 29 + 89 = 118
41) 73 + 74 = 147
42) 21 + 55 = 76
43) 19 + 41 = 60
44) 84 + 20 = 104
45) 37 + 33 = 70
46) 98 + 87 = 185
47) 24 + 88 = 112
48) 15 + 38 = 53
49) 70 + 66 = 136
50) 13 + 70 = 83
51) 26 + 84 = 110
52) 91 + 56 = 147
53) 80 + 17 = 97
54) 56 + 6 = 62
55) 73 + 60 = 133
56) 62 + 49 = 111
57) 70 + 37 = 

Lab 9 Exercise 5 — timed copy

In [7]:
%%writefile ex5_vecmul_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>


double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

#define N 10000000
#define THREADS_PER_BLOCK 512

__global__ void vecMul(int *a, int *b, int *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n) {
        c[i] = a[i] * b[i];
    }
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    size_t size = (size_t)N * sizeof(int);
    double dStart, dElaps;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size);
    b = (int *)malloc(size);
    c = (int *)malloc(size);

    if (a == NULL || b == NULL || c == NULL) {
        printf("Host memory allocation failed\n");
        return 1;
    }

    random_ints(a, N);
    random_ints(b, N);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;
    printf("Launching %d blocks x %d threads\n",
           blocks, THREADS_PER_BLOCK);

    dStart = cpuSecond();
    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess) {
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));
        return 1;
    }

    err = cudaDeviceSynchronize();
    if (err != cudaSuccess) {
        printf("Kernel execution failed: %s\n", cudaGetErrorString(err));
        return 1;
    }
    dElaps = cpuSecond() - dStart;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int i = N - 1000; i < N; i++)
        printf("%d) %d x %d = %d\n", i, a[i], b[i], c[i]);

    int errors = 0;
    for (int i = 0; i < N; i++) {
        if (c[i] != a[i] * b[i])
            errors++;
    }

    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a);
    free(b);
    free(c);
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}

Writing ex5_vecmul_timed.cu


In [8]:
!nvcc -arch=sm_75 ex5_vecmul_timed.cu -o ex5_vecmul_timed

In [21]:
!./ex5_vecmul_timed

Launching 19532 blocks x 512 threads
9999000) 76 x 52 = 3952
9999001) 76 x 74 = 5624
9999002) 41 x 94 = 3854
9999003) 8 x 0 = 0
9999004) 1 x 41 = 41
9999005) 28 x 52 = 1456
9999006) 45 x 4 = 180
9999007) 63 x 35 = 2205
9999008) 89 x 97 = 8633
9999009) 10 x 71 = 710
9999010) 74 x 91 = 6734
9999011) 65 x 41 = 2665
9999012) 29 x 60 = 1740
9999013) 53 x 5 = 265
9999014) 35 x 88 = 3080
9999015) 18 x 12 = 216
9999016) 17 x 12 = 204
9999017) 8 x 65 = 520
9999018) 51 x 34 = 1734
9999019) 79 x 64 = 5056
9999020) 13 x 89 = 1157
9999021) 10 x 85 = 850
9999022) 7 x 2 = 14
9999023) 31 x 51 = 1581
9999024) 88 x 86 = 7568
9999025) 59 x 8 = 472
9999026) 94 x 50 = 4700
9999027) 58 x 94 = 5452
9999028) 5 x 53 = 265
9999029) 36 x 67 = 2412
9999030) 80 x 45 = 3600
9999031) 33 x 5 = 165
9999032) 64 x 41 = 2624
9999033) 21 x 91 = 1911
9999034) 93 x 6 = 558
9999035) 65 x 34 = 2210
9999036) 1 x 43 = 43
9999037) 90 x 62 = 5580
9999038) 80 x 21 = 1680
9999039) 90 x 92 = 8280
9999040) 1 x 86 = 86
9999041) 54 x 1

Lab 9 Exercise 6 — timed copy

In [10]:
%%writefile ex6_matrix_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>


double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

#define ROWS 10000
#define COLS 10000

__global__ void mulElem(int *a, int *b, int *c, int rows, int cols) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    if (row < rows && col < cols) {
        int idx = row * cols + col;
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, size_t size) {
    for (size_t i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    double dStart, dElaps;

    size_t count = (size_t)ROWS * COLS;
    size_t size = count * sizeof(int);

    a = (int *)malloc(size);
    b = (int *)malloc(size);
    c = (int *)malloc(size);

    if (a == NULL || b == NULL || c == NULL) {
        printf("Host memory allocation failed\n");
        free(a);
        free(b);
        free(c);
        return 1;
    }

    cudaError_t err;

    err = cudaMalloc((void **)&d_a, size);
    if (err != cudaSuccess) {
        printf("GPU allocation failed: %s\n", cudaGetErrorString(err));
        free(a); free(b); free(c);
        return 1;
    }

    err = cudaMalloc((void **)&d_b, size);
    if (err != cudaSuccess) {
        printf("GPU allocation failed: %s\n", cudaGetErrorString(err));
        cudaFree(d_a);
        free(a); free(b); free(c);
        return 1;
    }

    err = cudaMalloc((void **)&d_c, size);
    if (err != cudaSuccess) {
        printf("GPU allocation failed: %s\n", cudaGetErrorString(err));
        cudaFree(d_a); cudaFree(d_b);
        free(a); free(b); free(c);
        return 1;
    }

    random_ints(a, count);
    random_ints(b, count);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(32, 16);

    dim3 numBlocks(
        (COLS + threadsPerBlock.x - 1) / threadsPerBlock.x,
        (ROWS + threadsPerBlock.y - 1) / threadsPerBlock.y
    );

    printf("Grid: %d x %d blocks\n", numBlocks.x, numBlocks.y);

    dStart = cpuSecond();
    mulElem<<<numBlocks, threadsPerBlock>>>(
        d_a, d_b, d_c, ROWS, COLS
    );

    err = cudaGetLastError();
    if (err != cudaSuccess) {
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));
        cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
        free(a); free(b); free(c);
        return 1;
    }

    err = cudaDeviceSynchronize();
    if (err != cudaSuccess) {
        printf("Kernel execution failed: %s\n", cudaGetErrorString(err));
        cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
        free(a); free(b); free(c);
        return 1;
    }
    dElaps = cpuSecond() - dStart;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (size_t i = count - 1000; i < count; i++) {
        printf("C[%zu][%zu] = %d x %d = %d\n",
               i / COLS, i % COLS, a[i], b[i], c[i]);
    }

    size_t errors = 0;

    for (size_t i = 0; i < count; i++) {
        if (c[i] != a[i] * b[i])
            errors++;
    }

    printf("Verification: %zu mismatches out of %zu\n",
           errors, count);
    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);

    return 0;
}

Writing ex6_matrix_timed.cu


In [11]:
!nvcc -arch=sm_75 ex6_matrix_timed.cu -o ex6_matrix_timed

In [22]:
!./ex6_matrix_timed

Grid: 313 x 625 blocks
C[9999][9000] = 29 x 56 = 1624
C[9999][9001] = 58 x 69 = 4002
C[9999][9002] = 76 x 50 = 3800
C[9999][9003] = 68 x 21 = 1428
C[9999][9004] = 67 x 78 = 5226
C[9999][9005] = 98 x 11 = 1078
C[9999][9006] = 3 x 29 = 87
C[9999][9007] = 69 x 52 = 3588
C[9999][9008] = 93 x 64 = 5952
C[9999][9009] = 60 x 65 = 3900
C[9999][9010] = 49 x 77 = 3773
C[9999][9011] = 5 x 44 = 220
C[9999][9012] = 6 x 77 = 462
C[9999][9013] = 17 x 31 = 527
C[9999][9014] = 45 x 65 = 2925
C[9999][9015] = 17 x 50 = 850
C[9999][9016] = 59 x 2 = 118
C[9999][9017] = 68 x 4 = 272
C[9999][9018] = 70 x 28 = 1960
C[9999][9019] = 54 x 11 = 594
C[9999][9020] = 7 x 87 = 609
C[9999][9021] = 29 x 93 = 2697
C[9999][9022] = 12 x 88 = 1056
C[9999][9023] = 14 x 26 = 364
C[9999][9024] = 74 x 6 = 444
C[9999][9025] = 24 x 52 = 1248
C[9999][9026] = 4 x 14 = 56
C[9999][9027] = 75 x 76 = 5700
C[9999][9028] = 0 x 40 = 0
C[9999][9029] = 65 x 7 = 455
C[9999][9030] = 67 x 13 = 871
C[9999][9031] = 29 x 96 = 2784
C[9999][9032] 

| Lab 9 program | Launch configuration | Run 1 (s) | Run 2 (s) | Run 3 (s) |
|---|---|---:|---:|---:|
| Exercise 3 | 512 blocks × 1 thread | 0.0007191 | 0.0001001 | 0.0001040 |
| Exercise 4 | 1 block × 512 threads | 0.0001218 | 0.0001311 | 0.0001011 |
| Exercise 5 | 10,000,000 elements; 19,532 blocks × 512 threads | 0.0006330 | 0.0006239 | 0.0006709 |
| Exercise 6 | 100,000,000 elements; 313 × 625 blocks; 32 × 16 threads | 0.0050769 | 0.0050349 | 0.0050559 |

Exercise 1(c) — Kernel time versus data-transfer time

In [23]:
%%writefile ex5_vecmul_100m_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>


double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

#define N 100000000
#define THREADS_PER_BLOCK 512

__global__ void vecMul(int *a, int *b, int *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n) {
        c[i] = a[i] * b[i];
    }
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    size_t size = (size_t)N * sizeof(int);
    double totalStart, kernelStart, kernelTime, totalTime;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size);
    b = (int *)malloc(size);
    c = (int *)malloc(size);

    if (a == NULL || b == NULL || c == NULL) {
        printf("Host memory allocation failed\n");
        return 1;
    }

    random_ints(a, N);
    random_ints(b, N);

    totalStart = cpuSecond();
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;
    printf("Launching %d blocks x %d threads\n",
           blocks, THREADS_PER_BLOCK);

    kernelStart = cpuSecond();
    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess) {
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));
        return 1;
    }

    err = cudaDeviceSynchronize();
    if (err != cudaSuccess) {
        printf("Kernel execution failed: %s\n", cudaGetErrorString(err));
        return 1;
    }
    kernelTime = cpuSecond() - kernelStart;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);
    totalTime = cpuSecond() - totalStart;

    for (int i = N - 1000; i < N; i++)
        printf("%d) %d x %d = %d\n", i, a[i], b[i], c[i]);

    int errors = 0;
    for (int i = 0; i < N; i++) {
        if (c[i] != a[i] * b[i])
            errors++;
    }

    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Kernel-only time - %.7f\n", kernelTime);
    printf("Whole sequence time - %.7f\n", totalTime);

    free(a);
    free(b);
    free(c);
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}

Writing ex5_vecmul_100m_timed.cu


In [24]:
!nvcc -arch=sm_75 ex5_vecmul_100m_timed.cu -o ex5_vecmul_100m_timed && ./ex5_vecmul_100m_timed

Launching 195313 blocks x 512 threads
99999000) 29 x 56 = 1624
99999001) 58 x 69 = 4002
99999002) 76 x 50 = 3800
99999003) 68 x 21 = 1428
99999004) 67 x 78 = 5226
99999005) 98 x 11 = 1078
99999006) 3 x 29 = 87
99999007) 69 x 52 = 3588
99999008) 93 x 64 = 5952
99999009) 60 x 65 = 3900
99999010) 49 x 77 = 3773
99999011) 5 x 44 = 220
99999012) 6 x 77 = 462
99999013) 17 x 31 = 527
99999014) 45 x 65 = 2925
99999015) 17 x 50 = 850
99999016) 59 x 2 = 118
99999017) 68 x 4 = 272
99999018) 70 x 28 = 1960
99999019) 54 x 11 = 594
99999020) 7 x 87 = 609
99999021) 29 x 93 = 2697
99999022) 12 x 88 = 1056
99999023) 14 x 26 = 364
99999024) 74 x 6 = 444
99999025) 24 x 52 = 1248
99999026) 4 x 14 = 56
99999027) 75 x 76 = 5700
99999028) 0 x 40 = 0
99999029) 65 x 7 = 455
99999030) 67 x 13 = 871
99999031) 29 x 96 = 2784
99999032) 75 x 28 = 2100
99999033) 44 x 64 = 2816
99999034) 97 x 69 = 6693
99999035) 42 x 59 = 2478
99999036) 94 x 75 = 7050
99999037) 53 x 99 = 5247
99999038) 64 x 11 = 704
99999039) 87 x 92